## Setup and Library Installation

To process complex musical data and prepare it for deep learning architectures, we used specialized audio and data processing libraries.

In [0]:
# Required libraries

%pip install pretty_midi music21 seaborn kaggle pandas matplotlib tqdm

## Imports and Data Ingestion

The dataset for this project originates from the Kaggle "MIDI Classic Music" corpus. To facilitate seamless collaboration across different cloud environments (Google Colab and Databricks) without requiring local API keys, the dataset has been hosted on a shared Google Drive. This cell uses the `gdown` library to securely download and extract the raw MIDI files into the local cluster's temporary storage.

In [0]:
# Imports
%pip install gdown
import gdown
import os
import zipfile

# Define paths
data_dir = "./data/midi_data"
os.makedirs(data_dir, exist_ok=True)
zip_path = os.path.join(data_dir, "midi-classic-music.zip")

# Download from Google Drive
file_id = '1QJPbVi6q2QPIiD47ZBf9tDxhM4ON-7mj'
url = f'https://drive.google.com/file/d/1QJPbVi6q2QPIiD47ZBf9tDxhM4ON-7mj/view?usp=share_link'

print("Downloading dataset from Google Drive...")
gdown.download(url, zip_path, quiet=False)

# Unzip the file
print(f"\nExtracting {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(data_dir)
    
print(f"Extraction complete! Files are located in: {data_dir}")

In [0]:
import os
print(os.listdir("./data/midi_data"))

In [0]:
import os

target_composers = ["Bach", "Beethoven", "Chopin", "Mozart"]
data_dir = "./data/midi_data"

for composer in target_composers:
    composer_path = os.path.join(data_dir, composer)
    if os.path.exists(composer_path):
        file_count = len([f for f in os.listdir(composer_path) if f.endswith('.mid')])
        print(f"{composer}: {file_count} MIDI files")
    else:
        print(f"{composer}: folder not found — check exact name")

## Data Filtering

The original corpus contains compositions from 175 different composers. Per the project requirements, our predictive models will focus on classifying four specific classical masters: **Bach, Beethoven, Chopin, and Mozart**. 

This script iterates through the extracted directory, retains only the target composers, and removes the remaining directories to optimize storage and memory usage.

In [0]:
# Filter for specific composers
import os
import shutil

target_composers = ['bach', 'beethoven', 'chopin', 'mozart']

# Find where the composer folders are
base_midi_path = data_dir
if "midi_classic_music" in os.listdir(data_dir):
    base_midi_path = os.path.join(data_dir, "midi_classic_music")

all_folders = [f for f in os.listdir(base_midi_path) if os.path.isdir(os.path.join(base_midi_path, f))]
print(f"Found {len(all_folders)} total composer folders.")

# Delete non-target folders
removed_count = 0
for folder in all_folders:
    # Use lowercase for safer string matching
    if folder.lower() not in target_composers:
        shutil.rmtree(os.path.join(base_midi_path, folder))
        removed_count += 1

print(f"Removed {removed_count} folders. Kept target composers only.")

## Data Cleaning

Raw data scraped from the internet frequently contains corrupted, truncated, or empty files. Feeding invalid files into deep learning pipelines causes fatal runtime errors during feature extraction. 

This cell acts as a quality control gateway: it attempts to parse every `.mid` and `.midi` file using `pretty_midi`, automatically deleting files that are structurally unreadable or contain zero musical notes.

Before exploring or training on the data, we must address class imbalance. If one composer has 1,000 files and another has only 200, our CNN and LSTM models may simply learn to always predict the majority class. 

This step calculates the size of the smallest composer class and randomly undersamples the majority classes to match it. A fixed random seed (`42`) is utilized to guarantee that all team members generate the exact same balanced dataset.

In [0]:
# Data Cleaning Pipeline
import pretty_midi
import glob
from tqdm import tqdm
import random
import os
from collections import Counter

# Find all .mid and .midi files in the remaining folders
midi_files = glob.glob(f"{base_midi_path}/**/*.mid", recursive=True) + \
             glob.glob(f"{base_midi_path}/**/*.midi", recursive=True)

print(f"Found {len(midi_files)} potential MIDI files for our 4 composers.")

valid_files = []
corrupt_files = []

# Loop and validate
for file_path in tqdm(midi_files, desc="Validating MIDI files"):
    try:
        # Parse the MIDI file
        midi_data = pretty_midi.PrettyMIDI(file_path)
        
        # Check if the file contains instruments/notes
        if len(midi_data.instruments) == 0:
            raise ValueError("No instruments found.")
            
        total_notes = sum(len(instrument.notes) for instrument in midi_data.instruments)
        if total_notes == 0:
            raise ValueError("No notes found.")
            
        # Add to valid list if it passes
        valid_files.append(file_path)
        
    except Exception as e:
        corrupt_files.append((file_path, str(e)))
        # Delete the bad file:
        os.remove(file_path)

print(f"\nCleaning Summary")
print(f"Valid Files: {len(valid_files)}")
print(f"Corrupt/Empty Files Removed: {len(corrupt_files)}")
if corrupt_files:
    print(f"Example error: {corrupt_files[0]}")

# Handling Class Imbalance via Undersampling
# Group valid files by composer
composer_files = {composer: [] for composer in target_composers}
composer_counts = Counter()

for file_path in valid_files:
    # Extract composer name from the folder path and lowercase it
    composer = os.path.basename(os.path.dirname(file_path)).lower()
    if composer in target_composers:
        composer_files[composer].append(file_path)
        composer_counts[composer] += 1
        
print("Original Clean File Counts")
for comp, count in composer_counts.items():
    print(f"{comp.capitalize()}: {count}")
    
# Minimum class size
min_class_size = min(composer_counts.values())
print(f"\nBalancing dataset to {min_class_size} files per composer")

# Randomly undersample the majority classes
balanced_valid_files = []

# Random seed so everyone gets the exact same files every time it runs
random.seed(42) 

for composer, files in composer_files.items():
    # Randomly select 'min_class_size' files from this composer's list
    sampled_files = random.sample(files, min_class_size)
    balanced_valid_files.extend(sampled_files)
    
print(f"\nBalanced Dataset Summary")
print(f"Total Balanced Files: {len(balanced_valid_files)}")

# Overwrite the original valid_files list 
valid_files = balanced_valid_files    

## Exploratory Data Analysis (EDA)

Before transforming the MIDI files into high-dimensional sequences (for the LSTM) or 2D piano-roll matrices (for the CNN), we extract high-level statistical features to understand the underlying distributions. 

This script loops through the balanced dataset and extracts metrics such as composition length, note density (notes per second), estimated tempo (BPM), and average pitch.

In [0]:
# EDA metrics
import pandas as pd
import numpy as np

eda_data = []

for file_path in tqdm(valid_files, desc="Extracting EDA Features"):
    try:
        composer = os.path.basename(os.path.dirname(file_path)).capitalize()
        midi_data = pretty_midi.PrettyMIDI(file_path)
        
        # Total length of piece in seconds
        length_sec = midi_data.get_end_time()
        
        # Total number of notes across all tracks
        total_notes = sum(len(instrument.notes) for instrument in midi_data.instruments)
        
        # Estimated Tempo (BPM)
        tempo = midi_data.estimate_tempo()
        
        # Average pitch and pitch range
        all_pitches = []
        for instrument in midi_data.instruments:
            # Ignoring drum tracks for this analysis
            if not instrument.is_drum:
                all_pitches.extend([note.pitch for note in instrument.notes])
                
        if all_pitches:
            avg_pitch = np.mean(all_pitches)
            pitch_range = np.max(all_pitches) - np.min(all_pitches)
        else:
            avg_pitch = 0
            pitch_range = 0
            
        # Append to list
        eda_data.append({
            "Composer": composer,
            "Length_Seconds": length_sec,
            "Total_Notes": total_notes,
            "Notes_Per_Second": total_notes / length_sec if length_sec > 0 else 0,
            "Estimated_BPM": tempo,
            "Average_Pitch": avg_pitch,
            "Pitch_Range": pitch_range
        })
    except Exception:
        continue # Skip files that act up during deep extraction

df_eda = pd.DataFrame(eda_data)
display(df_eda.head())

## Visualization

Using the extracted features, this cell generates a series of visualizations to explore structural differences between the four composers. 

By analyzing note density, piece length, and tempo distributions, we can visually identify if certain composers have distinct stylistic signatures that our neural networks will likely rely on for classification.

In [0]:
# Plotting the Data
import matplotlib.pyplot as plt
import seaborn as sns

# Visual style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 10))

# Distribution of Classes (Imbalance check)
plt.subplot(2, 2, 1)
sns.countplot(data=df_eda, x="Composer", palette="viridis")
plt.title("Number of Pieces per Composer")

# Notes per Second (Complexity/Density)
plt.subplot(2, 2, 2)
sns.boxplot(data=df_eda, x="Composer", y="Notes_Per_Second", palette="viridis")
plt.title("Note Density (Notes per Second) by Composer")
plt.ylim(0, df_eda["Notes_Per_Second"].quantile(0.95)) 

# Piece Lengths
plt.subplot(2, 2, 3)
sns.violinplot(data=df_eda, x="Composer", y="Length_Seconds", palette="viridis")
plt.title("Distribution of Piece Lengths (Seconds)")
plt.ylim(0, 1000)

# Estimated Tempo
plt.subplot(2, 2, 4)
sns.kdeplot(data=df_eda, x="Estimated_BPM", hue="Composer", fill=True, palette="viridis", common_norm=False)
plt.title("Tempo (BPM) Density by Composer")

plt.tight_layout()
plt.show()

In [0]:
# LSTM Feature Extraction
# Converts each MIDI file into a sequence of note/chord "tokens" using music21.
# Each token represents either a single note (pitch) or a chord (multiple pitches
# played together), which the LSTM will learn sequential patterns from.

from music21 import converter, note, chord
from tqdm import tqdm

def extract_note_sequence(file_path):
    """Parse a MIDI file and return a list of note/chord tokens."""
    midi = converter.parse(file_path)
    notes_to_parse = None

    parts = midi.getElementsByClass('Part')
    if parts:
        notes_to_parse = parts[0].recurse()
    else:
        notes_to_parse = midi.flat.notes

    sequence = []
    for element in notes_to_parse:
        if isinstance(element, note.Note):
            sequence.append(str(element.pitch))
        elif isinstance(element, chord.Chord):
            sequence.append('.'.join(str(n) for n in element.normalOrder))

    return sequence

# Run extraction across the balanced dataset
lstm_data = []  # list of (composer, sequence) tuples
failed_files = []

for file_path in tqdm(valid_files, desc="Extracting note sequences for LSTM"):
    try:
        composer = os.path.basename(os.path.dirname(file_path)).capitalize()
        sequence = extract_note_sequence(file_path)
        if len(sequence) > 0:
            lstm_data.append((composer, sequence))
    except Exception as e:
        failed_files.append((file_path, str(e)))
        continue

print(f"\nSuccessfully extracted sequences from {len(lstm_data)} files")
print(f"Failed to extract: {len(failed_files)} files")
if failed_files:
    print(f"Example failure: {failed_files[0]}")

In [0]:
# LSTM Sequence Encoding
# Builds a vocabulary of all unique note/chord tokens seen across the dataset,
# then converts each sequence into a list of integers the LSTM can consume.

from collections import Counter

# Build vocabulary from all sequences
all_tokens = [token for (composer, sequence) in lstm_data for token in sequence]
token_counts = Counter(all_tokens)
vocab = sorted(token_counts.keys())

token_to_int = {token: i for i, token in enumerate(vocab)}
int_to_token = {i: token for token, i in token_to_int.items()}

vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size} unique note/chord tokens")

# Encode each sequence as a list of integers
encoded_data = [(composer, [token_to_int[t] for t in sequence]) for composer, sequence in lstm_data]

# Quick sanity check
print(f"\nExample encoded sequence (first 20 tokens) from a {encoded_data[0][0]} piece:")
print(encoded_data[0][1][:20])

# Check sequence length distribution (important for choosing a fixed window size next)
seq_lengths = [len(seq) for (_, seq) in encoded_data]
print(f"\nSequence length stats:")
print(f"Min: {min(seq_lengths)}, Max: {max(seq_lengths)}, Mean: {sum(seq_lengths)/len(seq_lengths):.0f}")

In [0]:
# LSTM Training Data Preparation
# Since sequence lengths vary hugely (29 to 5210 tokens), we slice each piece
# into fixed-length windows. Each window becomes one training example, labeled
# with its composer. This also multiplies our effective training data.

import numpy as np

SEQUENCE_LENGTH = 100  # number of tokens per training example

X = []
y = []

composers_list = sorted(set(composer for composer, _ in encoded_data))
composer_to_int = {c: i for i, c in enumerate(composers_list)}

for composer, seq in encoded_data:
    label = composer_to_int[composer]
    # Slide a window across the sequence with 50% overlap (stride = SEQUENCE_LENGTH // 2)
    stride = SEQUENCE_LENGTH // 2
    for start in range(0, max(1, len(seq) - SEQUENCE_LENGTH + 1), stride):
        window = seq[start:start + SEQUENCE_LENGTH]
        if len(window) == SEQUENCE_LENGTH:
            X.append(window)
            y.append(label)

X = np.array(X)
y = np.array(y)

print(f"Composer label mapping: {composer_to_int}")
print(f"Total training windows generated: {len(X)}")
print(f"X shape: {X.shape}, y shape: {y.shape}")

# Check class balance after windowing
unique, counts = np.unique(y, return_counts=True)
for label_idx, count in zip(unique, counts):
    composer_name = composers_list[label_idx]

In [0]:
%pip install tensorflow

In [0]:
# Train/Test Split + LSTM Model
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Split into train/test (stratified to preserve composer balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# One-hot encode labels for multi-class classification
num_classes = len(composers_list)
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

# Build the LSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=SEQUENCE_LENGTH),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [0]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# One-hot encode labels for multi-class classification
num_classes = len(composers_list)
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

# Build the LSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=SEQUENCE_LENGTH),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [0]:
# Train the LSTM model
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=30,
    batch_size=64,
    verbose=1
)

In [0]:
# LSTM Model Evaluation
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

# Predict on test set
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Overall accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}\n")

# Precision, Recall, F1 per composer
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=composers_list))

# Confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Visualize confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=composers_list, yticklabels=composers_list)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('LSTM Confusion Matrix')
plt.tight_layout()
plt.show()